# Delta Lake com Apache Spark

Este notebook demonstra como usar Delta Lake com PySpark para criar uma tabela transacional local.

- Criação e escrita de tabela Delta
- Leitura e consulta de dados
- Atualização de registros
- Time travel

In [ ]:
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = configure_spark_with_delta_pip(SparkSession.builder)
spark = (
    builder.appName("DeltaLakeStudy")
           .master("local[*]")
           .getOrCreate()
)

delta_path = os.path.abspath("tmp/delta_table")
os.makedirs(os.path.dirname(delta_path), exist_ok=True)

data = [(1, "Alice", 1200), (2, "Bob", 1500), (3, "Carol", 1600)]
df = spark.createDataFrame(data, ["id", "name", "salary"])

df.write.format("delta").mode("overwrite").save(delta_path)
print(f"Tabela Delta escrita em: {delta_path}")

In [ ]:
spark.read.format("delta").load(delta_path).show()

spark.sql("DROP TABLE IF EXISTS delta_demo")
spark.sql(f"CREATE TABLE delta_demo USING DELTA LOCATION '{delta_path}'")
spark.sql("SELECT * FROM delta_demo").show()

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)
delta_table.update("id = 1", {"salary": "salary + 500"})
print("Registro atualizado para id = 1")

spark.read.format("delta").load(delta_path).show()

In [ ]:
print("Versão 0 - antes da atualização")
spark.read.format("delta").option("versionAsOf", 0).load(delta_path).show()